# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook guides users through exploring the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. The dataset contains records describing clinical and pathological variables for cancer survivors with second primary colorectal cancer (CRC).

### Dataset Source
The dataset is defined via a Croissant schema URL:
`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json

# Define the dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata and records
dataset = mlc.Dataset(croissant_url)

# Display metadata overview
metadata = dataset.metadata.to_json()
print(f"Dataset Title: {metadata['name']}")
print(f"Description: {metadata['description']}")
print(f"Authors (@id): {[author['@id'] for author in metadata['author']]}")
print(f"Keywords: {metadata['keywords']}")

## 2. Data Overview

Review available **record sets**, their **fields/columns**, and their `@id`s. All subsequent references use these IDs.

In [ ]:
# List available record sets in the dataset
record_sets = dataset.record_sets
print("Available record sets:")
for rec_set in record_sets:
    print(f"  RecordSet @id: {rec_set['@id']}  |  name: {rec_set.get('name', 'N/A')}")

if len(record_sets) == 0:
    print("No record sets found in 'recordSet' property. Attempting alternative extraction...")
    # Try to infer from distribution
    distributions = metadata.get('distribution', [])
    for dist in distributions:
        print(f"Distribution file @id: {dist['@id']}")
    print("You can use these distribution @id as source for records.")

# Explore fields for (first) record set or distribution
fields_info = {}
rs_ids = [rec_set['@id'] for rec_set in record_sets]
for rs_id in rs_ids:
    fields = dataset.fields(record_set=rs_id)
    print(f"\nFields in RecordSet {rs_id}:")
    for fld in fields:
        print(f"  Field @id: {fld['@id']}  |  name: {fld.get('name', 'N/A')}")
    fields_info[rs_id] = fields

# If no recordSet, try distribution (for schema compatibility)
if len(record_sets) == 0:
    distributions = metadata.get('distribution', [])
    for dist in distributions:
        dist_id = dist['@id']
        try:
            fields = dataset.fields(record_set=dist_id)
            print(f"\nFields in Distribution {dist_id}:")
            for fld in fields:
                print(f"  Field @id: {fld['@id']}  |  name: {fld.get('name', 'N/A')}")
            fields_info[dist_id] = fields
        except Exception as exc:
            print(f"Could not access fields for Distribution {dist_id}")

## 3. Data Extraction

Load records from each record set/table using their `@id` and then convert to DataFrame for analysis. Record sets and field ids are referenced using `@id` for all operations.

If record sets are unavailable, extract from distributions.

In [ ]:
# Prepare set of entity IDs for extraction

dataframes = {}
target_ids = []

# Prefer record sets, fallback to distributions if not present
if len(record_sets) > 0:
    target_ids = [rs['@id'] for rs in record_sets]
else:
    # Use distribution @id for record extraction
    distributions = metadata.get('distribution', [])
    target_ids = [dist['@id'] for dist in distributions]

for entity_id in target_ids:
    try:
        recs = list(dataset.records(record_set=entity_id))
        dataframes[entity_id] = pd.DataFrame(recs)
        print(f"DataFrame for '{entity_id}' loaded: {dataframes[entity_id].shape[0]} rows, {dataframes[entity_id].shape[1]} columns.")
        print("  Columns:", dataframes[entity_id].columns.tolist())
    except Exception as exc:
        print(f"Could not load records for '{entity_id}': {exc}")

# Preview the first DataFrame
if len(dataframes) > 0:
    first_id = list(dataframes.keys())[0]
    print(f"\nPreview of records from '{first_id}':")
    display(dataframes[first_id].head())

## 4. Exploratory Data Analysis (EDA)

Perform typical processing steps: filter, normalize, categorize, and group using column `@id`s. Substitute actual field/column names by their `@id`s where available.

In [ ]:
# Choose record set to analyze (use the first loaded)
record_set_id = list(dataframes.keys())[0]
df = dataframes[record_set_id]

# Identify numeric fields (@id) through available metadata
fields = dataset.fields(record_set=record_set_id)
numeric_field_id = None
for fld in fields:
    if fld.get('dataType', '').lower() in ['integer', 'float', 'number']:
        numeric_field_id = fld['@id']
        print(f"Selected numeric field @id: {numeric_field_id} (name: {fld.get('name','')})")
        break

# If no numeric field found with @id, fallback to known variables
if numeric_field_id is None:
    # Try common field names
    common_numeric = [col for col in df.columns if 'age' in col.lower() or 'interval' in col.lower() or 'msi' in col.lower()]
    if len(common_numeric) > 0:
        numeric_field_id = common_numeric[0]
        print(f"Auto-selected numeric field: {numeric_field_id}")

# Filtering example: select records with numeric_field > threshold
threshold = 10
if numeric_field_id is not None and numeric_field_id in df.columns:
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"\nFiltered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalization example
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping example: find a categorical group field
    group_field_id = None
    for fld in fields:
        if fld.get('dataType', '').lower() == 'text':
            group_field_id = fld['@id']
            print(f"Using group field @id: {group_field_id} (name: {fld.get('name','')})")
            break
    if group_field_id is None:
        # Try common column names
        for col in df.columns:
            if 'sex' in col.lower() or 'location' in col.lower() or 'msi' in col.lower():
                group_field_id = col
                print(f"Auto-selected group field: {group_field_id}")
                break
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped {numeric_field_id} mean by {group_field_id}:")
        display(grouped_df.head())
else:
    print("No valid numeric field found for EDA.")

## 5. Visualization

Visualize distributions and relationships between selected variables using their `@id`.

In [ ]:
# Visualize numeric distribution and group comparison
if numeric_field_id is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id], kde=True, bins=15)
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id is not None and group_field_id in df.columns:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"'{numeric_field_id}' grouped by '{group_field_id}'")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion

This notebook demonstrated loading, overview, processing, and visualization of the FAIR^2 colorectal cancer dataset using `mlcroissant`. By referencing all dataset elements via their `@id`, users can consistently extract and manipulate variables. Further exploration may involve advanced statistical modeling or deep analysis of MSI status and anatomical distributions for cancer survivors.

**Summary:**
- Dataset loaded and described via Croissant schema.
- Record sets, fields, and columns explored and extracted.
- EDA applied: filtering, normalization, grouping, all by `@id`.
- Visualizations illustrated key distributions.

For more detail, consult the [mlcroissant documentation](https://github.com/mlcommons/croissant) and FAIR^2 dataset guidelines.